In [1]:
import logging
from pathlib import Path

import pandas as pd
from _pandas_patcher import PrintAsImageString
from IPython.display import HTML, display
from rdkit import Chem
from stereomolgraph import StereoMolGraph
from stereomolgraph.algorithms.symmetry import ext_sym_num

logging.getLogger("_pandas_patcher").setLevel(logging.ERROR)

In [2]:
smiles = [
    "CC",
    "CC(C)(C)C",
    "CC(C)(C)C(F)(F)F",
    "C1CC12CC2",
    "C[C@@H]1[C@@H](C)C21[C@@H](C)[C@@H]2C",
    "C[C@@H]1[C@@H](C)C21[C@@H](C)[C@H]2C",
    "[H]C([H])(C12CCC(C(F)([H])[H])(CC2)CC1)F",
    "I[C@](F)([H])C1(CC2)CCC2([C@](I)(F)[H])CC1",
    "I[C@](F)([H])C1(CC2)CCC2([C@](F)(I)[H])CC1",
]

In [3]:
rows = []
for smile in smiles:
    rdmol = Chem.MolFromSmiles(smile)
    if rdmol is not None:
        rdmol = Chem.AddHs(rdmol)

    if rdmol is None:
        rows.append(
            {
                "stereomolgraph": None,
                "smiles": smile,
                "ext_sym_num_upper_bound": None,
            }
        )
        continue

    graph = StereoMolGraph.from_rdmol(
        rdmol, stereo_complete=True, resonance=True, lone_pair_stereo=True
    )

    rows.append(
        {
            "stereomolgraph": graph,
            "smiles": smile,
            "ext_sym_num_upper_bound": ext_sym_num(graph),
        }
    )

df_symmetry = pd.DataFrame(rows)
if not df_symmetry.empty:
    df_symmetry["ext_sym_num_upper_bound"] = df_symmetry[
        "ext_sym_num_upper_bound"
    ].astype("Int64")

In [ ]:
styled = (
    df_symmetry[["stereomolgraph", "smiles", "ext_sym_num_upper_bound"]]
    .style.hide(axis="index")
    .format(
        {
            "stereomolgraph": PrintAsImageString,
        },
        escape=None,
    )
    .set_properties(**{"text-align": "left", "vertical-align": "middle"})
    .set_table_styles(
        [
            {"selector": "table", "props": [("border-collapse", "collapse")]},
            {
                "selector": "th:not(:last-child), td:not(:last-child)",
                "props": [("border-right", "1px solid #b0b0b0")],
            },
            {"selector": "th", "props": [("text-align", "left")]},
        ]
    )
)

table_html = styled.to_html()
html_document = (
    "<!DOCTYPE html><html lang='en'><head><meta charset='utf-8'>"
    "<title>External Symmetry</title></head><body>"
    f"<div style='overflow-x:auto'>{table_html}</div></body></html>"
)
project_root = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").exists()
)
output_html_path = project_root / "reports" / "external_symmetry.html"
output_html_path.parent.mkdir(parents=True, exist_ok=True)
output_html_path.write_text(html_document, encoding="utf-8")

display(HTML(f"<div style='overflow-x:auto'>{table_html}</div>"))
print(f"Saved HTML report to: {output_html_path.resolve()}")

Saved HTML report to: C:\Users\maxim\Code\experiments\SI_SymmetryNumbers\external_symmetry.html
